# Resume PDF Matching

A notebook for extracting and matching details from the resume PDF at `c:\Users\admin\Downloads\John_Lester_Mayuga_CV.pdf`.

This notebook will:
1. Import required libraries
2. Load the PDF resume
3. Extract text from the CV
4. Parse resume sections
5. Match extracted details to target criteria
6. Display matching results


In [ ]:
import re
from pathlib import Path

try:
    import PyPDF2
except ImportError:
    raise ImportError('PyPDF2 is required for PDF extraction. Install with `pip install PyPDF2`.')

import pandas as pd


## Load PDF Resume

Open the specified CV PDF file and prepare it for text extraction.


In [ ]:
pdf_path = Path(r"C:\Users\admin\Downloads\John_Lester_Mayuga_CV.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(f"Resume PDF not found: {pdf_path}")

print('Resume PDF path:', pdf_path)


## Extract Text from CV PDF

Use PDF parsing tools to extract the raw text content from the resume file.


In [ ]:
def extract_pdf_text(path: Path) -> str:
    reader = PyPDF2.PdfReader(str(path))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text() or ""
        print(f"Extracted text from page {page_number}: {len(page_text)} chars")
        pages.append(page_text)
    return "\n\n".join(pages)

raw_text = extract_pdf_text(pdf_path)
print('\n--- Resume Text Preview ---\n')
print(raw_text[:2500])


## Parse Resume Sections

Split and parse resume text into structured sections like education, experience, and skills.


In [ ]:
def parse_resume_sections(text: str) -> dict:
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    sections = {}
    current_section = 'Summary'
    buffer = []
    section_headers = ['Education', 'Experience', 'Skills', 'Projects', 'Certifications', 'Awards', 'Contact', 'Summary', 'Professional Experience', 'Technical Skills']

    for line in lines:
        normalized = line.rstrip(':').strip()
        if any(normalized.lower() == header.lower() for header in section_headers):
            if buffer:
                sections[current_section] = '\n'.join(buffer)
                buffer = []
            current_section = normalized
        else:
            buffer.append(line)

    if buffer:
        sections[current_section] = '\n'.join(buffer)

    return sections

parsed_sections = parse_resume_sections(raw_text)
for name, content in parsed_sections.items():
    print(f"\n## {name}\n")
    print(content[:600])


## Match Extracted Details to Target Criteria

Compare parsed resume details against predefined criteria or job requirements.


In [ ]:
criteria = {
    'experience': ['Next.js', 'React', 'TypeScript', 'API', 'Web', 'Node.js', 'frontend', 'portfolio'],
    'skills': ['React', 'Next.js', 'TypeScript', 'JavaScript', 'CSS', 'HTML', 'API', 'Git'],
    'education': ['Bachelor', 'Computer Science', 'Engineering', 'Bachelors', 'BS', 'College']
}


def match_criteria(sections: dict, criteria: dict) -> dict:
    results = {}
    combined_text = ' '.join(sections.values()).lower()

    for category, terms in criteria.items():
        matches = [term for term in terms if term.lower() in combined_text]
        results[category] = {
            'matched': matches,
            'score': len(matches),
            'total': len(terms)
        }

    return results

match_results = match_criteria(parsed_sections, criteria)
match_results


## Display Matching Results

Summarize and present the matched details with a score or highlights.


In [ ]:
result_rows = []
for category, data in match_results.items():
    result_rows.append({
        'Category': category.title(),
        'Matched Terms': ', '.join(data['matched']) or 'None',
        'Score': f"{data['score']}/{data['total']}"
    })

results_df = pd.DataFrame(result_rows)
results_df
